# B1 · Rilevatore di palla morta (DFL)

Obiettivo: stimare per ogni frame la probabilità che la palla sia **morta** (`p_dead`), così da
scrivere `D` dove il gioco è davvero fermo e non semplicemente dove il tracking ha perso la palla.

Tutto qui gira sulle **7 partite DFL** (Bassek et al. 2025, CC BY 4.0: altre partite, quindi
training e tuning sono permessi). La logica è in `pipeline/deadball.py`; il notebook la esegue e
spiega i risultati. Nessun dato della partita fornita viene letto.

In [ ]:
import os, sys
REPO = os.environ.get("MTAH_REPO") or next((p for p in ["/kaggle/input/hackaton-barcellona-code", "/kaggle/working/Hackaton_Barcellona", os.path.abspath("..")] if os.path.isdir(os.path.join(p, "pipeline"))), None)
assert REPO, "Imposta MTAH_REPO alla cartella della repo"
sys.path[:0] = [REPO, os.path.join(REPO, "pipeline"), os.path.join(REPO, "starter"), os.path.join(REPO, "evaluation")]

## 1. Perché "palla mancante" non è "palla morta"

Il baseline scrive `D` dove il tracking non vede la palla. Sul nostro tracking reale la palla manca
nel 41% dei frame valutati, quasi sempre a gioco fermo, **ma non sempre**:

- una palla persa per un secondo durante il gioco diventa una `D` falsa, che **spezza un possesso in
  due** (due possessi predetti contro uno vero: la spellF1 perde un match e guadagna un falso positivo);
- uno stop in cui la palla resta tracciata (rimessa veloce, punizione con la palla ferma) è una `D`
  mancata, che **fonde due possessi**.

La spellF1 accoppia i possessi per IoU temporale, quindi ogni errore sul confine vivo/morto pesa
molto più di quanto dica l'accuratezza per frame. Serve un rilevatore che guardi anche **cosa fanno
i giocatori**.

Le condizioni di input sono simulate come le nostre (seed fissi): palla da
`dfl_degrade.metrica_like_ball` (manca nel 90% degli stop > 1 s, più buchi brevi nel 2% del gioco),
giocatori con rumore 0.2 m e 1% di scambi di squadra del portatore (come `dfl_method4.py`), più il 5%
di frame-giocatore cancellati a episodi di 1-6 s (i nostri slot hanno buchi). **Si addestra sui
dati degradati**, non su quelli puliti.

In [ ]:
import numpy as np, pandas as pd
import dfl_paths, deadball
pd.set_option("display.width", 200)
paths = dfl_paths.match_paths()
print(len(paths), "partite in", dfl_paths.converted_dir())
print("output in", dfl_paths.out_dir(), "| modelli in", deadball.models_dir())

## 2. Le feature

Riferimenti:

- **Lang et al. 2022** (Scientific Reports, *Predicting the in-game status in soccer with machine
  learning using spatiotemporal player tracking data*): solo posizioni dei giocatori, statistiche
  collettive con shift temporali di ±0.4, 0.8, 2, 4, 20, 40, 80 s, AdaBoost (meglio di random forest
  e regressione logistica), ~92% di accuratezza per frame, F1 0.80 sugli stop ≥ 2 s con IoU ≥ 0.5,
  filtro mediano di 3.64 s. Le feature più importanti sono il contesto entro circa ±4 s, con più peso
  al **passato**; il 77-81% degli stop trovati ha inizio e fine entro ±2 s (split 45/10/47 partite).
- **Borghesi et al. 2023** (ablation): per in gioco / morta contano soprattutto la **velocità** e il
  flag **fuori campo**; con la sola palla si arriva all'83.6% di accuratezza. Quindi flag fuori
  campo sia per la palla sia per i giocatori.
- **Vidal-Codina et al. 2022**: durante i dead-ball interval (calcio d'inizio, rimessa, corner) gli
  **schieramenti** sono riconoscibili: giocatori nelle rispettive metà, giocatori in area, vicino
  alla bandierina.

Cosa calcola `deadball.features(d, ball, players, team)` per ogni frame (metà tempo per metà tempo,
così nessuna finestra attraversa l'intervallo):

| gruppo | feature |
|---|---|
| movimento | n. giocatori visti, velocità media / mediana / p10 / p90 / max, quota che cammina (< 1 m/s), corre (> 4), sprinta (> 7), accelerazione media, coerenza (quanto si muovono nella stessa direzione), velocità per squadra |
| forma | baricentro per squadra (distanza dal centrocampo), distanza tra baricentri, dispersione per squadra e totale, estensione in x e y |
| schieramento | giocatori fuori campo (n., flag, distanza massima), distanza minima di un giocatore da bandierina / centro / dischetto, n. nel cerchio di centrocampo, n. massimo in un'area di rigore, quota di giocatori nella metà della propria squadra |
| palla (quando c'è) | presenza, velocità, palla ferma, distanza fuori dal campo e flag (> 0.5 m), distanza da bandierine, centro, dischetti, area piccola, distanza dal giocatore più vicino |
| assenza palla | durata del buco in corso, da quanto manca, tra quanto torna, durata del tratto visto in corso |
| contesto | 14 feature collettive mediate su finestre **passate e future** di 0.4, 0.8, 2, 4, 20, 40, 80 s (medie mobili con cumsum, NaN-safe) |

Le velocità sono calcolate per oggetto (slot), non per identità: posizioni lisciate con
**Savitzky-Golay** (polinomio 2, finestra 7 frame) e derivata SG; velocità > 12 m/s (impossibili:
scambio di slot o salto del tracker) diventano NaN. Così le stesse feature funzionano sul nostro
tracking reale, con slot senza identità e buchi.

In [ ]:
d = dict(np.load(paths[0]))
ball, p, t = deadball.degraded_inputs(d, seed=0)
X = deadball.features(d, ball, p, t)
names = deadball.feature_names()
print(X.shape, "| palla mancante nel {:.1%} dei frame, palla morta (vera) nel {:.1%}".format(
    np.isnan(ball[:, 0]).mean(), (d["alive"] == 0).mean()))
pd.DataFrame(X[::250, :45], columns=names[:45]).describe().T.round(2)

## 3. Modello, leave-one-match-out e post-processing

- **LightGBM** binario (parametri fissi, `deterministic=True`, `num_threads` fisso, seed 0);
  se `lightgbm` non c'è, `HistGradientBoostingClassifier` di sklearn. Al posto dell'AdaBoost di Lang
  et al.: stesso principio (boosting di alberi), più veloce.
- **Leave-one-match-out**: ogni partita è predetta da un modello addestrato sulle altre 6
  (`p_dead` out-of-fold). In training si usa un frame ogni `stride` (default 5): frame vicini sono
  quasi identici.
- **Filtro mediano** su `p_dead` (finestre 1, 51, 91 = 3.64 s, 151 frame) e **soglia**
  (0.3-0.7), scelti in modo annidato: per ogni partita tenuta fuori si sceglie la coppia che
  massimizza la spellF1 media **sulle altre 6**. I numeri riportati non sono quindi ottimizzati
  sulla partita che misurano.
- **Durata minima dello stop** 25 frame (1 s): nei dati DFL le interruzioni ≤ 1 s sono quasi
  assenti (2 su 711), quindi una `D` più breve è quasi sempre rumore che spezzerebbe un possesso.

Il calcolo completo (7 addestramenti + griglia + modello finale) richiede alcuni minuti.

In [ ]:
res = deadball.run(stride=5, long_shifts=True)

## 4. Come leggere le metriche

- `frame_acc`: frazione di frame con vivo/morto giusto. `f1_dead`: F1 della classe morta per frame.
  Sono le metriche di Lang et al. (~92% di accuratezza) ma **non** quelle che contano per noi.
- `stop_f1`: gli stop (veri e predetti) ≥ 2 s sono oggetti, accoppiati 1-a-1 per IoU ≥ 0.5 (greedy
  per IoU decrescente); F1 di quell'accoppiamento. Confrontabile con lo 0.80 di Lang et al.
- `macro_f1`, `spell_f1`: **effetto a valle**, con `dfl_bench.score` (la metrica della gara). Le
  etichette H/A vengono sempre dallo stesso baseline (giocatore più vicino alla palla degradata,
  voto di maggioranza ±50 frame); cambia solo **dove si scrive D**:
  - `ball missing (baseline)`: D dove manca la palla;
  - `detector`: D dove `p_dead` filtrata > soglia;
  - `missing & detector`: D solo dove la palla manca **e** il rilevatore dice morta;
  - `oracle D (gt)`: D vera, il tetto raggiungibile cambiando solo la D.

  Nei frame dichiarati vivi ma senza palla per più di 2 s la squadra è quella dell'ultimo possesso.
  Differenze di spellF1 sotto ~0.04 sono rumore (EVALUATION.md).

In [ ]:
res["summary"].round(3)

In [ ]:
res["table"].pivot_table(index="match", columns="variant", values="spell_f1").round(3)

Griglia (spellF1 media sulle 7 partite, prima della scelta annidata): righe variante × finestra,
colonne soglia. Una superficie piatta vuol dire che la scelta è robusta.

In [ ]:
res["grid"].round(3)

## 5. Importanza delle feature

Quota del guadagno (gain) totale, media sui 7 fold (min e max accanto). Ci si aspetta, come in Lang
et al., che dominino le statistiche di velocità con contesto di pochi secondi, e che le finestre
passate pesino più di quelle future.

In [ ]:
res["importance"].head(20).round(4) if res["importance"] is not None else "importanza non disponibile (fallback sklearn)"

## 6. Un tratto di partita

`p_dead` filtrata (out-of-fold) contro la verità DFL (`alive`) e contro la palla mancante del
baseline. Si vedono i due errori del baseline: buchi della palla a gioco vivo (D false) e stop con
palla visibile (D mancate).

In [ ]:
import matplotlib.pyplot as plt
name = next(iter(res["per_match"]))
m = res["per_match"][name]
a = len(m["alive"]) // 4; z = a + 25 * 600          # 10 minuti
tt = np.arange(a, z) / 25 / 60
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(tt, 0, 1, where=m["missing"][a:z], color="0.85", step="mid", label="palla mancante")
ax.plot(tt, 1 - m["alive"][a:z], color="black", lw=1.2, drawstyle="steps-mid", label="palla morta (vera)")
ax.plot(tt, m["p_dead"][a:z], color="tab:red", lw=1, label="p_dead filtrata (out-of-fold)")
ax.axhline(m["threshold"], color="tab:red", ls=":", lw=1, label="soglia {:.2f}".format(m["threshold"]))
ax.set_xlabel("minuti dall'inizio del file"); ax.set_ylim(-0.05, 1.05)
ax.set_title("{} — finestra mediana {} frame".format(name, m["window"]))
ax.legend(loc="upper right", ncol=4, fontsize=8)
plt.show()

## 7. Output

- `dfl_paths.out_path(match, "deadball")` → `<match>_deadball.npz` con `p_dead` (F,) float32
  out-of-fold e già filtrata, `variant`, `threshold`, `window`, `min_dead`: è l'input per A1/B2.
- `models/deadball_lgbm.txt` (o `deadball_hgb.joblib`) + `models/deadball_meta.json`: modello finale
  addestrato su tutte e 7 le partite, lista delle feature e post-processing scelto. Su Kaggle sono in
  `/kaggle/working/models`.

Per la partita fornita (solo in locale, da un umano): costruire `ball` (F,2) e `players` (F,P,2)
normalizzati con NaN, `team` per slot e `half`, poi
`p, dead, meta = deadball.predict_p_dead(ball, players, team, half, folder="models")`.

In [ ]:
for name in list(res["per_match"])[:3]:
    z = np.load(dfl_paths.out_path(name, "deadball"))
    print(name, z["p_dead"].shape, str(z["variant"]))
print("modello finale:", res["final"])